In [1]:
import pandas as pd
data = pd.read_csv(r"C:\Users\Admin\OneDrive\Documents\price_volume_data.csv", header=[0, 1], index_col=0, parse_dates=True)
data.head()

Price      Adj Close                                                         \
Ticker          AAPL ABBV        ABT       ADBE       ADSK   AMD       AMGN   
Date                                                                          
2010-01-04  6.424605  NaN  18.414782  37.090000  25.670000  9.70  39.592060   
2010-01-05  6.435713  NaN  18.266008  37.700001  25.280001  9.71  39.249092   
2010-01-06  6.333344  NaN  18.367441  37.619999  25.340000  9.57  38.954151   
2010-01-07  6.321635  NaN  18.519604  36.889999  25.480000  9.47  38.597458   
2010-01-08  6.363664  NaN  18.614286  36.689999  26.260000  9.43  38.940434   

Price                                     ... Volume                     \
Ticker        AMZN        APD       ASML  ...   UBER       UL       UNH   
Date                                      ...                             
2010-01-04  6.6950  51.808655  32.425728  ...    NaN   799200  12199500   
2010-01-05  6.7345  51.378773  32.678314  ...    NaN   929500  11180700   
2010-01-06  6.6125  50.955124  32.977688  ...    NaN  1811600   9761100   
2010-01-07  6.5000  50.662312  32.060867  ...    NaN  1365300  11789800   
2010-01-08  6.6760  50.986294  31.293720  ...    NaN   761700   7228700   

Price                                                                      
Ticker           UPS         V       WFC       WMT       XOM XYZ      YUM  
Date                                                                       
2010-01-04   3897200  20180000  39335700  62259300  27809100 NaN  2962274  
2010-01-05   5966300  25833600  55416000  46945200  30174700 NaN  3298757  
2010-01-06   5770200  16254000  33237000  37551600  35044700 NaN  4178981  
2010-01-07   5747000  27841200  61649000  31988100  27192100 NaN  2452472  
2010-01-08  13779300  11907200  35508700  34089600  24891800 NaN  3772392  

[5 rows x 600 columns]

In [2]:
import matplotlib.pyplot as plt

In [3]:
returns = data['Adj Close'].pct_change()
returns.columns = pd.MultiIndex.from_product([['Return'], returns.columns])
data_with_returns = pd.concat([data, returns], axis=1)
data_with_returns.head()

Adj Close                                                         \
Ticker          AAPL ABBV        ABT       ADBE       ADSK   AMD       AMGN   
Date                                                                          
2010-01-04  6.424605  NaN  18.414782  37.090000  25.670000  9.70  39.592060   
2010-01-05  6.435713  NaN  18.266008  37.700001  25.280001  9.71  39.249092   
2010-01-06  6.333344  NaN  18.367441  37.619999  25.340000  9.57  38.954151   
2010-01-07  6.321635  NaN  18.519604  36.889999  25.480000  9.47  38.597458   
2010-01-08  6.363664  NaN  18.614286  36.689999  26.260000  9.43  38.940434   

                                          ... Return                      \
Ticker        AMZN        APD       ASML  ...   UBER        UL       UNH   
Date                                      ...                              
2010-01-04  6.6950  51.808655  32.425728  ...    NaN       NaN       NaN   
2010-01-05  6.7345  51.378773  32.678314  ...    NaN -0.021793 -0.001586   
2010-01-06  6.6125  50.955124  32.977688  ...    NaN -0.005411  0.009848   
2010-01-07  6.5000  50.662312  32.060867  ...    NaN -0.007040  0.038377   
2010-01-08  6.6760  50.986294  31.293720  ...    NaN  0.004834 -0.009391   

                                                                            
Ticker           UPS         V       WFC       WMT       XOM XYZ       YUM  
Date                                                                        
2010-01-04       NaN       NaN       NaN       NaN       NaN NaN       NaN  
2010-01-05  0.001719 -0.011459  0.027452 -0.009958  0.003904 NaN -0.003420  
2010-01-06 -0.007379 -0.013428  0.001425 -0.002235  0.008643 NaN -0.007148  
2010-01-07 -0.007606  0.009307  0.036286  0.000560 -0.003142 NaN -0.000288  
2010-01-08  0.048075  0.002766 -0.009269 -0.005037 -0.004012 NaN  0.000288  

[5 rows x 700 columns]

In [4]:
from scipy.stats.mstats import winsorize

In [5]:
class PrepareData:
    def __init__(self, file):
        self.file = file
        self.df = pd.read_csv(self.file, header=[0, 1], index_col=0, parse_dates=True)
        self.tickers = set(self.df.columns.get_level_values(1))

    def add_returns(self):
        returns = self.df['Adj Close'].pct_change()
        returns.columns = pd.MultiIndex.from_product([['Return'], returns.columns])
        self.df = pd.concat([self.df, returns], axis=1)

    def check_na(self, ticker):
        print(f"\nTicker: {ticker}")
        for col in self.tickers:
            print(f"{col} : {self.df[col, ticker].isna().sum()}")
            
    def add_missing_dates(self):
        expected_dates = pd.date_range(start=data_with_returns.index.min(),
                               end=data_with_returns.index.max(),
                               freq='B')  # 'B' = business days

        #find missing dates 
        missing_dates = expected_dates.difference(data_with_returns.index)
        print(len(missing_dates))
        print("Missing dates:\n", missing_dates)

        #fill data 
        #cac cot con lai: NaN
        data_filled = data_with_returns.reindex(expected_dates)

        #lay data truoc do de diền gia tri NaN
        data_filled = data_filled.ffill()

    def detect_outliers(self, ticker, col):
        series = self.df[(col, ticker)]
        Q1 = series.quantile(0.25)
        Q3 = series.quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = series[(series < lower_bound) | (series > upper_bound)]
        return outliers
    
    def draw_boxplot(self, ticker, col):
        series = self.df[(col, ticker)]
        plt.figure()
        plt.boxplot(series.dropna())
        plt.title(f"Boxplot of {col} for {ticker}")
        plt.show()
    
    def draw_histogram(self, ticker, col):
        series = self.df[(col, ticker)]
        plt.figure()
        plt.hist(series, bins=50)
        plt.title(f"Return of {ticker}")
        plt.show()

    def winsorize_data(self): 
        for ticker in self.tickers:
            returns = self.df["Return", ticker]
            self.df["Return", ticker] = winsorize(returns, limits=[0.05, 0.05])

            vol = self.df["Volume", ticker]
            self.df["Volume", ticker] = winsorize(vol, limits=[0.05, 0.05])

    def save_cleaned_data(self, output_file):
        self.df.to_csv(output_file)
    

prepare = PrepareData(r"C:\Users\Admin\OneDrive\Documents\price_volume_data.csv")
prepare.add_returns()
# for ticker in tickers:
#     prepare.check_na(ticker)

# for ticker in tickers:
#     prepare.draw_boxplot(ticker, 'Return')

# for ticker in tickers:
#     prepare.draw_histogram(ticker, 'Return')

In [6]:

prepare.add_missing_dates()
prepare.winsorize_data()
prepare.save_cleaned_data(r"C:\Users\Admin\OneDrive\Documents\cleaned_price_volume_data.csv")

147
Missing dates:
 DatetimeIndex(['2010-01-18', '2010-02-15', '2010-04-02', '2010-05-31',
               '2010-07-05', '2010-09-06', '2010-11-25', '2010-12-24',
               '2011-01-17', '2011-02-21',
               ...
               '2024-12-25', '2025-01-01', '2025-01-09', '2025-01-20',
               '2025-02-17', '2025-04-18', '2025-05-26', '2025-06-19',
               '2025-07-04', '2025-09-01'],
              dtype='datetime64[ns]', length=147, freq=None)


In [7]:
print(set(prepare.df.columns.get_level_values(0)))

{'Open', 'Adj Close', 'Low', 'Volume', 'High', 'Close', 'Return'}


In [8]:
import numpy as np

In [9]:

class Calculate:
    def __init__ (self, file):
        self.file = file
        self.df = pd.read_csv(self.file, header=[0, 1], index_col=0, parse_dates=True)
        self.tickers = set(self.df.columns.get_level_values(1))

    def add_SMA(self, short_window = 50, long_window = 200):
        SMA_50 = self.df['Adj Close'].rolling(window=short_window).mean()
        SMA_200 = self.df['Adj Close'].rolling(window=long_window).mean()
        SMA_50.columns = pd.MultiIndex.from_product([['SMA_50'], self.tickers])
        SMA_200.columns = pd.MultiIndex.from_product([['SMA_200'], self.tickers])
        self.df = pd.concat([self.df, SMA_50, SMA_200], axis=1)

    def calculate_position(self):
        target_risk = 0.01  # 1% of portfolio value
        h_l = self.df['High'] - self.df['Low']
        prev_close = self.df['Adj Close'].shift(1)
        h_pl = (self.df['High'] - prev_close).abs()
        l_pl = (self.df['Low'] - prev_close).abs()
        tr = np.maximum.reduce([h_l, h_pl, l_pl])
        tr = pd.DataFrame(tr, index=self.df.index, columns=self.df['High'].columns)
        atr = tr.rolling(window=20).mean()
        position_size = (target_risk / (atr / self.df['Close']))
        position_size.columns = pd.MultiIndex.from_product([['Position Size'], position_size.columns])
        self.df = pd.concat([self.df, position_size], axis=1)

c = Calculate(r"C:\Users\Admin\OneDrive\Documents\cleaned_price_volume_data.csv")
c.add_SMA()
c.calculate_position()
print(set(c.df.columns.get_level_values(0)))
c.df.tail()


{'Open', 'Adj Close', 'Low', 'Volume', 'SMA_200', 'Position Size', 'High', 'Close', 'SMA_50', 'Return'}


Adj Close                                                  \
                  AAPL        ABBV         ABT        ADBE        ADSK   
Date                                                                     
2025-10-06  256.690002  228.542419  133.147934  350.140015  323.429993   
2025-10-07  256.480011  231.163513  132.431122  348.309998  314.190002   
2025-10-08  258.059998  229.584900  133.675583  348.769989  311.410004   
2025-10-09  254.039993  229.038834  132.719818  347.470001  310.320007   
2025-10-10  245.270004  228.850189  131.983109  337.510010  303.500000   

                                                                         ...  \
                   AMD        AMGN        AMZN         APD         ASML  ...   
Date                                                                     ...   
2025-10-06  203.710007  294.119995  220.899994  270.950012  1041.452271  ...   
2025-10-07  211.509995  295.540009  221.779999  270.890015  1000.524902  ...   
2025-10-08  235.559998  294.619995  225.220001  268.579987   986.060547  ...   
2025-10-09  232.889999  295.429993  227.740005  262.709991   978.803406  ...   
2025-10-10  214.899994  290.130005  216.369995  257.079987   934.531982  ...   

           Position Size                                                    \
                    UBER        UL       UNH       UPS         V       WFC   
Date                                                                         
2025-10-06      0.352620  0.867209  0.348898  0.527367  0.588410  0.478872   
2025-10-07      0.338672  0.901692  0.406663  0.511786  0.593050  0.506176   
2025-10-08      0.343678  0.905637  0.405057  0.506459  0.610000  0.490344   
2025-10-09      0.321097  0.935397  0.416842  0.520159  0.594094  0.495595   
2025-10-10      0.309502  0.900596  0.388727  0.471522  0.574378  0.453785   

                                                    
                 WMT       XOM       XYZ       YUM  
Date                                                
2025-10-06  0.644291  0.565627  0.306052  0.607265  
2025-10-07  0.621179  0.564386  0.307543  0.580036  
2025-10-08  0.637349  0.567546  0.322422  0.586348  
2025-10-09  0.619227  0.550109  0.320331  0.572248  
2025-10-10  0.610552  0.528418  0.267759  0.555917  

[5 rows x 1000 columns]